# 🎬 Train FrameMind Cinematography LoRA Adapters
### Free-tier Google Colab (NVIDIA T4 GPU) — 100% Free Pipeline

This notebook trains domain-specialized **LoRA (Low-Rank Adaptation)** adapter weights for the **FrameMind** Edge AI Cinematography System.

* **Base Model:** `unsloth/gemma-2-2b-it`
* **Hardware:** Free Google Colab T4 GPU ($0 cost)
* **Output:** ~3.1 MB `adapter_model.safetensors`
* **Domain Adapters:** Podcast & Banter LoRA, Fitness Form LoRA, Keynote Presentation LoRA.

In [ ]:
# Step 1: Install Unsloth & Lightweight Fine-Tuning Dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# Step 2: Load Base Gemma-2 2B Model in 4-bit Precision
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✓ Base Gemma-2 2B loaded into T4 VRAM.")

In [ ]:
# Step 3: Attach Low-Rank Attention Adapters (Rank = 8)
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 864,
)
print("✓ LoRA attention adapters attached.")

In [ ]:
# Step 4: Load Synthetic Cinematography Dataset
import json
from datasets import Dataset

with open('synthetic_framing_dataset.json', 'r') as f:
    raw_data = json.load(f)

formatted = []
for entry in raw_data:
    prompt = f"<start_of_turn>user\nEvaluate cinematography intent for:\nTranscript: {entry['transcript']}\nAcoustic Cues: {', '.join(entry['acoustic_cues'])}<end_of_turn>\n<start_of_turn>model\n{{\"shot\": \"{entry['target_shot']}\", \"zoom\": {entry['target_zoom']}, \"panX\": {entry['target_pan_x']}, \"rationale\": \"{entry['rationale']}\" }}<end_of_turn>"
    formatted.append({"text": prompt})

dataset = Dataset.from_list(formatted)
print(f"✓ Loaded {len(dataset)} training samples.")

In [ ]:
# Step 5: Train LoRA with SFTTrainer (Supervised Fine-Tuning)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)
trainer.train()
print("✓ Training complete in ~4 minutes on T4 GPU.")

In [ ]:
# Step 6: Export Lightweight LoRA Safetensors (~3.1 MB)
model.save_pretrained("framemind_lora_adapter")
tokenizer.save_pretrained("framemind_lora_adapter")

!ls -lh framemind_lora_adapter/adapter_model.safetensors
print("✓ Exported adapter_model.safetensors ready for on-device deployment.")